# 📘 Week 9: Biomedical Signals (ECG & EEG) Preprocessing
Welcome to Week 9! In this module, we will explore processing techniques for **Biomedical Signals**—specifically focusing on noise removal in **Electrocardiogram (ECG)** and frequency band isolation in **Electroencephalogram (EEG)**.

## 🎯 Learning Objectives:
- Simulate synthetic ECG waveforms using Gaussian curve modeling.
- Identify and remove typical biomedical signal noise: baseline wander and powerline interference.
- Design and apply High-Pass and Notch filters to clean corrupted ECG data.
- Simulate and analyze EEG signals, using band-pass filters to isolate key brainwave frequencies (Delta, Theta, Alpha, Beta).

## 1. Simulating an Electrocardiogram (ECG) Waveform
An ECG measures the electrical activity of the heart. A typical heartbeat consists of a sequence of waves labeled alphabetically: **P, Q, R, S, and T**.
- **P Wave**: Atrial depolarization.
- **QRS Complex**: Ventricular depolarization (characterized by the sharp R peak).
- **T Wave**: Ventricular repolarization.

We can model these individual peaks as Gaussian curves of varying amplitudes ($a$), positions ($b$), and widths ($c$):
$$G(t) = a \cdot e^{-\frac{(t - b)^2}{2 c^2}}$$

Let's write a function to simulate multiple continuous heartbeats (e.g. at 72 beats per minute, which is 1.2 Hz, or approximately 0.83 seconds per beat).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

fs = 500  # 500 Hz sampling rate
duration = 5.0  # 5 seconds duration
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

def generate_single_heartbeat(fs_rate=500):
    """
    Generates a single heartbeat cycle (0.8 seconds at 72 bpm)
    """
    t_beat = np.linspace(0, 0.83, int(fs_rate * 0.83), endpoint=False)
    
    # Gaussian peak definitions for P, Q, R, S, T
    # amplitude, peak location (seconds), peak width
    peaks = [
        (0.12, 0.15, 0.015), # P
        (-0.15, 0.23, 0.005), # Q
        (1.10, 0.25, 0.005), # R
        (-0.30, 0.27, 0.005), # S
        (0.35, 0.45, 0.025)  # T
    ]
    
    heartbeat = np.zeros_like(t_beat)
    for a, b, c in peaks:
        heartbeat += a * np.exp(- (t_beat - b)**2 / (2 * c**2))
        
    return heartbeat

# Repeat the single heartbeat to fill the duration of 5 seconds
single_beat = generate_single_heartbeat(fs)
num_repeats = int(duration / 0.83) + 1
clean_ecg = np.tile(single_beat, num_repeats)[:len(t)]

# Plot the simulated clean ECG
plt.figure(figsize=(12, 4))
plt.plot(t, clean_ecg, color='#1f77b4', linewidth=2)
plt.title("Simulated Clean ECG Signal (72 BPM)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Corrupting the ECG with Noise
ECG signals are typically contaminated by two major noise sources:
1. **Baseline Wander (Drift)**: Low-frequency noise caused by patient breathing or movement (typically < 0.5 Hz).
2. **Powerline Interference**: High-frequency noise from nearby AC electricity lines (typically exactly 50 Hz or 60 Hz depending on the country).

Let's add both to our clean ECG.

In [ ]:
# 1. Baseline Wander (slow respiration drift modeled as a 0.25 Hz sine wave)
drift = 0.4 * np.sin(2 * np.pi * 0.25 * t)

# 2. Powerline Noise (50 Hz sine wave)
powerline = 0.15 * np.sin(2 * np.pi * 50 * t)

# Corrupt signal
noisy_ecg = clean_ecg + drift + powerline

# Plot the corrupted waveform
plt.figure(figsize=(12, 4))
plt.plot(t, noisy_ecg, color='r', alpha=0.7)
plt.title("Corrupted ECG Signal (with Baseline Wander + 50 Hz Powerline Noise)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Filtering the ECG Signal
To retrieve the clean ECG signal, we apply two filters:
1. **High-Pass Filter**: A Butterworth high-pass filter with a cutoff frequency of 0.5 Hz to remove baseline wander.
2. **Notch Filter**: An IIR Notch filter centered at 50 Hz to remove powerline interference without modifying adjacent frequencies.

Let's implement and chain these filters.

In [ ]:
# 1. Remove Baseline Wander using High-Pass Filter
fc_hp = 0.5  # 0.5 Hz cutoff frequency
nyq = 0.5 * fs
b_hp, a_hp = signal.butter(2, fc_hp / nyq, btype='high')
# Apply filter forwards and backwards to avoid phase shift (zero-phase filtering)
ecg_no_drift = signal.filtfilt(b_hp, a_hp, noisy_ecg)

# 2. Remove Powerline Noise using IIR Notch Filter
f_notch = 50.0  # 50 Hz notch frequency
Q = 30.0  # Quality factor
b_notch, a_notch = signal.iirnotch(f_notch / nyq, Q)
filtered_ecg = signal.filtfilt(b_notch, a_notch, ecg_no_drift)

# Plot original, corrupted, and filtered ECG side-by-side
plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(t, clean_ecg, color='g', label='Clean ECG')
plt.legend()
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(t, noisy_ecg, color='r', label='Noisy ECG (Drift + 50Hz)')
plt.legend()
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(t, filtered_ecg, color='b', label='Filtered ECG')
plt.legend()
plt.xlabel("Time (seconds)")
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. EEG Signal Analysis and Brainwave Isolation
An EEG measures the brain's electrical activity. Unlike the heart, the brain generates continuous waveforms consisting of several overlapping bands:
- **Delta (0.5 - 4 Hz)**: Associated with deep sleep.
- **Theta (4 - 8 Hz)**: Associated with drowsiness, meditation, and light sleep.
- **Alpha (8 - 12 Hz)**: Prominent during relaxed wakefulness, especially with eyes closed.
- **Beta (12 - 30 Hz)**: Associated with active thinking, focus, or high alert.

Let's generate a synthetic EEG signal made of multiple bands and use a band-pass filter to isolate the **Alpha** band waves.

In [ ]:
fs_eeg = 250  # 250 Hz sampling rate
t_eeg = np.linspace(0, 4.0, int(fs_eeg * 4.0), endpoint=False)

# Generate synthetic brainwave bands
delta_wave = 0.4 * np.sin(2 * np.pi * 2 * t_eeg)      # 2 Hz (Delta)
alpha_wave = 0.8 * np.sin(2 * np.pi * 10 * t_eeg)     # 10 Hz (Alpha)
beta_wave = 0.25 * np.sin(2 * np.pi * 20 * t_eeg)     # 20 Hz (Beta)
eeg_noise = np.random.normal(0, 0.15, len(t_eeg))     # Background noise

# Combine to form EEG signal
eeg_signal = delta_wave + alpha_wave + beta_wave + eeg_noise

# Design Band-pass filter to isolate Alpha waves (8 - 12 Hz)
f_low = 8.0
f_high = 12.0
nyq_eeg = 0.5 * fs_eeg
b_bp, a_bp = signal.butter(3, [f_low / nyq_eeg, f_high / nyq_eeg], btype='bandpass')

# Apply filter
alpha_isolated = signal.filtfilt(b_bp, a_bp, eeg_signal)

# Plot the original EEG and the isolated Alpha band
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t_eeg, eeg_signal, color='k', label='Mixed EEG Signal')
plt.title("EEG Signal Preprocessing & Brainwave Isolation")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(t_eeg, alpha_wave, 'g--', label='Target Alpha Wave (10 Hz)', alpha=0.6)
plt.plot(t_eeg, alpha_isolated, color='g', label='Isolated Alpha Band (8-12 Hz)')
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## ✅ Reflection & Exercises
- **Filter order impact:** What would happen to the isolated Alpha waves if we decreased the Butterworth band-pass filter order from `3` to `1`? Test it and observe the leakage from Delta/Beta bands.
- **Notch Filter Q-factor:** How does the notch filter quality factor $Q$ affect the signal? What happens if you make $Q = 2$ instead of $Q = 30$? Observe if it removes part of the original ECG shape.